# Canvas 追加ラボ用: データセットを S3 にアップロードする

このノートブックは、追加ラボ [04_Canvas_ノーコード体験.md](../04_Canvas_ノーコード体験.md) で使う
`canvas_dataset.csv` を S3 にアップロードするためのものです。

Canvas は S3 上のデータを直接読み込めるため、ここで得られる `s3://...` のパスを
Canvas の **Import data → Tabular → Amazon S3** の **Input S3 endpoint** に貼り付けて使います。

> ℹ️ このノートブックは `workshop.ipynb`（プロコード経路）とは独立して単体で実行できます。


## 0. セットアップ

S3 バケットとアップロード用のクライアントを準備します。


In [ ]:
from sagemaker.core.helper.session_helper import Session
from sagemaker.core.s3 import S3Uploader

session = Session()
bucket = session.default_bucket()
prefix = "tokyo-port-wave-height"

print(f"bucket : {bucket}")
print(f"prefix : {prefix}")


## 1. データセットを用意する

`canvas_dataset.csv` が手元（このノートブックと同じフォルダ）にあれば、そのまま次のアップロードへ進めます。
無い場合は、以下のセルで `generate_synthetic_data.py` を実行して生成します（プロコード経路の擬似データ
生成スクリプトと同じもので、Canvas 用に列を絞らず全列を出力します）。


In [ ]:
import os

# canvas_dataset.csv が無ければ生成する（既にあればスキップ）
if not os.path.exists("canvas_dataset.csv"):
    print("canvas_dataset.csv が見つからないため生成します...")
    !python generate_synthetic_data.py --output canvas_dataset.csv --start-year 2021 --end-year 2025 --seed 123
else:
    print("canvas_dataset.csv は既に存在します。")


In [ ]:
import pandas as pd

# 内容を確認する（Canvas ラボでは全 11 列をそのまま使う）
canvas_df = pd.read_csv("canvas_dataset.csv")
print(f"行数: {len(canvas_df):,}")
print(f"列  : {list(canvas_df.columns)}")
canvas_df.head()


## 2. S3 にアップロードする

アップロード後に表示される `s3://...` のパスを控えて、Canvas の **Input S3 endpoint** に貼り付けてください。


In [ ]:
canvas_s3_uri = S3Uploader.upload("canvas_dataset.csv", f"s3://{bucket}/{prefix}/canvas")
print("Canvas で使う S3 パス:")
print(canvas_s3_uri)


## 3. 次のステップ

上で表示された `s3://...` のパスをコピーし、[04_Canvas_ノーコード体験.md](../04_Canvas_ノーコード体験.md) の
「1.3 Canvas を開いてフローを作成する」に進んでください。Canvas の **Import data → Tabular → Amazon S3**
の **Input S3 endpoint** にこのパスを貼り付けて **Go** を選択します。
